In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import time

# Import all the models we will test
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier, 
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

def run_classification(df, target_column, numerical_features, categorical_features):
    """
    A helper function to run a full classification test on a given target column.
    """
    print("===" * 15)
    print(f" STARTING TASK: PREDICTING '{target_column}'")
    print("===" * 15)
    
    # 1. Define Features (X) and Target (y)
    try:
        X = df.drop(target_column, axis=1)
        y = df[target_column]
    except KeyError:
        print(f"Error: Target column '{target_column}' not found.")
        return
    except Exception as e:
        print(f"An error occurred separating X and y: {e}")
        return

    # 2. Split Data (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Calculate Baseline Accuracy
    baseline_accuracy = y_train.value_counts(normalize=True).max()
    print(f"Target Variable: '{target_column}'")
    print(f"Class Distribution:\n{y_train.value_counts(normalize=True).to_string()}")
    print(f"\nBaseline Accuracy (Always guessing '{y_train.value_counts().idxmax()}'): {baseline_accuracy:.2%}")
    print("---" * 10)

    # 4. Create Preprocessing Pipelines
    # Transformer for numerical features
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    # Transformer for categorical features (outputs sparse matrix)
    categorical_transformer_sparse = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ])
    
    # Transformer for categorical features (outputs dense array for Naive Bayes)
    categorical_transformer_dense = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Preprocessor for most models
    preprocessor_sparse = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer_sparse, categorical_features)
        ],
        remainder='passthrough'
    )
    
    # Preprocessor for Naive Bayes
    preprocessor_dense = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer_dense, categorical_features)
        ],
        remainder='passthrough'
    )
    
    # 5. Define Models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
        "Support Vector Machine (SVM)": SVC(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
        "Gaussian Naive Bayes": GaussianNB()
    }

    # 6. Iterate, Train, and Score
    print("\n--- Model Test Results ---")
    results = {}
    for name, model in models.items():
        start_time = time.time()
        
        # Use the dense preprocessor for Naive Bayes, sparse for all others
        if name == "Gaussian Naive Bayes":
            clf_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor_dense),
                ('classifier', model)
            ])
        else:
            clf_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor_sparse),
                ('classifier', model)
            ])

        try:
            clf_pipeline.fit(X_train, y_train)
            y_pred = clf_pipeline.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            end_time = time.time()
            
            results[name] = accuracy
            print(f"  Model: {name:<25} | Accuracy: {accuracy:<6.2%} | Time: {end_time - start_time:.2f}s")
        
        except Exception as e:
            print(f"  Model: {name} - FAILED ({e})")
    
    # Print summary for this task
    best_model_name = max(results, key=results.get)
    best_accuracy = results[best_model_name]
    
    print("\n--- Task Summary ---")
    print(f"Best Model: {best_model_name}")
    print(f"Best Accuracy: {best_accuracy:.2%}")
    print(f"Baseline Accuracy: {baseline_accuracy:.2%}")
    print(f"Improvement over Baseline: {best_accuracy - baseline_accuracy:.2%}")
    print("===" * 15)
    print("\n\n")


def run_all_classification_tasks(file_path):
    """
    Main function to load and prep data, then run classification
    for both 'Customer type' and 'Gender'.
    """
    try:
        # 1. Load Data
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return
    except Exception as e:
        print(f"An error occurred loading the file: {e}")
        return

    print(f"Successfully loaded data from {file_path}\n")

    # 2. Drop Unnecessary Columns
    columns_to_drop = [
        'Invoice ID', 'gross margin percentage', 'Sales', 
        'Tax 5%', 'cogs', 'gross income', 'Rating'
    ]
    df = df.drop(columns=columns_to_drop, errors='ignore')

    # 3. Feature Engineering (Date/Time)
    try:
        df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
        df['Time'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p').dt.hour
    except ValueError as e:
        print(f"Warning: Could not parse Date/Time columns: {e}")
        print("Proceeding without date and time features.")
        base_numerical_features = ['Unit price', 'Quantity']
        base_categorical_features = ['Branch', 'City', 'Product line', 'Payment']
        df = df.drop(columns=['Date', 'Time'], errors='ignore')
    else:
        df['Month'] = df['Date'].dt.month
        df['DayOfWeek'] = df['Date'].dt.day_name()
        df['Hour'] = df['Time']
        df = df.drop(columns=['Date', 'Time'])
        base_numerical_features = ['Unit price', 'Quantity', 'Month', 'Hour']
        base_categorical_features = ['Branch', 'City', 'Product line', 'Payment', 'DayOfWeek']

    # --- TASK 1: PREDICT 'Customer type' ---
    # For this task, 'Gender' is a feature.
    cat_features_task1 = base_categorical_features + ['Gender']
    num_features_task1 = base_numerical_features
    run_classification(df.copy(), 'Customer type', num_features_task1, cat_features_task1)

    # --- TASK 2: PREDICT 'Gender' ---
    # For this task, 'Customer type' is a feature.
    cat_features_task2 = base_categorical_features + ['Customer type']
    num_features_task2 = base_numerical_features
    run_classification(df.copy(), 'Gender', num_features_task2, cat_features_task2)


# --- USER: PLEASE CHANGE THIS LINE ---
file_path_here = "/Users/nipunjuneja/knime-workspace/Supermarket_Sales/SuperMarket Analysis.csv"
# -------------------------------------

run_all_classification_tasks(file_path_here)

Successfully loaded data from /Users/nipunjuneja/knime-workspace/Supermarket_Sales/SuperMarket Analysis.csv

 STARTING TASK: PREDICTING 'Customer type'
Target Variable: 'Customer type'
Class Distribution:
Customer type
Member    0.565
Normal    0.435

Baseline Accuracy (Always guessing 'Member'): 56.50%
------------------------------

--- Model Test Results ---
  Model: Logistic Regression       | Accuracy: 56.00% | Time: 0.01s
  Model: Decision Tree             | Accuracy: 53.50% | Time: 0.01s
  Model: Random Forest             | Accuracy: 54.50% | Time: 0.10s
  Model: Gradient Boosting         | Accuracy: 49.00% | Time: 0.09s
  Model: K-Nearest Neighbors (KNN) | Accuracy: 56.00% | Time: 0.01s
  Model: Support Vector Machine (SVM) | Accuracy: 59.50% | Time: 0.03s
  Model: AdaBoost                  | Accuracy: 57.50% | Time: 0.05s
  Model: Hist Gradient Boosting    | Accuracy: 48.50% | Time: 1.13s
  Model: Gaussian Naive Bayes      | Accuracy: 51.00% | Time: 0.01s

--- Task Summary ---